# TRACE: Data Exploration & Incomplete Basemap Simulation

This notebook visualizes:
1. High-resolution satellite GeoTIFF tiles and OpenStreetMap GeoJSON linestrings.
2. Basemap degradation simulation strategies (Random Edge Dropout, Suburban Sprawl, Disaster Severance).
3. The 4-channel multi-modal tensor stack `[Red, Green, Blue, Prior_Map]`.



In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import rasterio
import geopandas as gpd

# Add project root to sys.path
sys.path.append("..")
from src.data.rasterizer import rasterize_linestrings
from src.data.degradation import simulate_incomplete_map
from src.common.geo_utils import geo_to_pixel


## 1. Load Satellite Tile & Vector Road Linestrings


In [ ]:
img_path = "../data/raw/synthetic_tile_001.tif"
geo_path = "../data/raw/synthetic_tile_001.geojson"

with rasterio.open(img_path) as src:
    img_rgb = src.read([1, 2, 3]).transpose(1, 2, 0)
    transform = src.transform
    crs = src.crs

gdf = gpd.read_file(geo_path)
print(f"Loaded imagery with shape: {img_rgb.shape}, CRS: {crs}")
print(f"Loaded {len(gdf)} vector road linestrings.")


## 2. Simulate Incomplete Prior Map (30% Road Dropout)


In [ ]:
prior_gdf, residual_gdf = simulate_incomplete_map(gdf, strategy="random_drop", drop_ratio=0.30, seed=42)
print(f"Known Prior Roads (70%): {len(prior_gdf)} segments")
print(f"Missing Residual Roads (30%): {len(residual_gdf)} segments (What TRACE must discover!)")


## 3. Visualize Ground Truth vs Incomplete Basemap


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Subplot 1: Raw Satellite Image
axes[0].imshow(img_rgb)
axes[0].set_title("1. Raw Satellite Imagery (RGB)")
axes[0].axis("off")

# Subplot 2: Known Prior Map (Channel 4)
axes[1].imshow(img_rgb)
for geom in prior_gdf.geometry:
    coords = [geo_to_pixel(x, y, transform) for x, y in geom.coords]
    xs, ys = zip(*coords)
    axes[1].plot(xs, ys, color="cyan", linewidth=3, label="Prior Map")
axes[1].set_title("2. Existing Incomplete Map (70% Prior)")
axes[1].axis("off")

# Subplot 3: Missing Roads (Ground Truth Target)
axes[2].imshow(img_rgb)
for geom in residual_gdf.geometry:
    coords = [geo_to_pixel(x, y, transform) for x, y in geom.coords]
    xs, ys = zip(*coords)
    axes[2].plot(xs, ys, color="red", linewidth=3, linestyle="--", label="Target Residual")
axes[2].set_title("3. Target Residual Roads (30% to Discover)")
axes[2].axis("off")

plt.tight_layout()
plt.show()


## 4. Inspect the 4-Channel Input Tensor Stack


In [ ]:
prior_mask = rasterize_linestrings(prior_gdf, shape=(512, 512), transform=transform, line_thickness_pixels=4, gaussian_sigma=1.5)
target_mask = rasterize_linestrings(residual_gdf, shape=(512, 512), transform=transform, line_thickness_pixels=4, gaussian_sigma=0.0)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(prior_mask, cmap="magma")
axes[0].set_title("Channel 4: Prior Map Mask (with Gaussian Tolerance)")
axes[0].axis("off")

axes[1].imshow(target_mask, cmap="inferno")
axes[1].set_title("Target Label: Residual Centerline Mask")
axes[1].axis("off")

plt.tight_layout()
plt.show()
